In [2]:
%pip install torch_geometric

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import cv2
import numpy as np
import os
from sklearn.neighbors import kneighbors_graph

# Define feature extraction function
def extract_features(image, img_size=128):
    if image is None or image.size == 0:
        raise ValueError("Empty image or invalid image format")
    resized_image = cv2.resize(image, (img_size, img_size))
    features = resized_image.flatten()
    return features

# Define function to create graph data with sparse edges using k-nearest neighbors
def create_graph_data(features, labels, k=8):
    x = torch.tensor(features, dtype=torch.float)  # Node features
    num_samples = len(features)
    k_neighbors = min(k, num_samples - 1)  # Ensure k_neighbors is valid
    if k_neighbors > 0:
        adj = kneighbors_graph(features, k_neighbors, mode='connectivity', include_self=False)
        edge_index = torch.tensor(np.array(adj.nonzero()), dtype=torch.long)
    else:
        edge_index = torch.tensor([[], []], dtype=torch.long)  # No edges
    y = torch.tensor(labels, dtype=torch.long)  # Labels
    data = Data(x=x, edge_index=edge_index, y=y)
    return data

# Define the GNN model
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.lin(x)
        return x

# Function to load dataset
def load_dataset(dataset_dir):
    image_paths = []
    labels = []
    for label_dir in os.listdir(dataset_dir):
        label_path = os.path.join(dataset_dir, label_dir)
        if os.path.isdir(label_path):
            for img_name in os.listdir(label_path):
                img_path = os.path.join(label_path, img_name)
                image_paths.append(img_path)
                labels.append(label_dir)
    return image_paths, labels

# Load and preprocess the data
dataset_dir = r"//folder path"
image_paths, labels = load_dataset(dataset_dir)

# Encode labels
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
img_size = 128
features = []
valid_labels = []  # Only keep labels for successfully loaded images
for img_path, label in zip(image_paths, labels_encoded):
    img_array = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img_array is not None:
        try:
            features.append(extract_features(img_array, img_size))
            valid_labels.append(label)
        except Exception as e:
            print(f"Error processing image at {img_path}: {e}")
    else:
        print(f"Warning: Unable to load image at {img_path}")

# Scale features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(features_scaled, valid_labels, test_size=0.2, random_state=42)

# Create graph data
train_data = create_graph_data(X_train, y_train, k=8)
test_data = create_graph_data(X_test, y_test, k=8)

# Create data loaders
train_loader = DataLoader([train_data], batch_size=1, shuffle=True)
test_loader = DataLoader([test_data], batch_size=1, shuffle=False)

# Initialize model, optimizer, and loss function
in_channels = X_train.shape[1]
hidden_channels = 64
out_channels = len(label_encoder.classes_)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNN(in_channels, hidden_channels, out_channels).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

# Training and testing functions
def train(model, optimizer, criterion, train_loader, device):
    model.train()
    correct = 0
    total = 0
    running_loss = 0.0

    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        # Ensure target is of the same shape as output
        target = data.y.view(-1)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(out, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        running_loss += loss.item()

    train_accuracy = correct / total
    average_loss = running_loss / len(train_loader)

    return train_accuracy, average_loss

def test(model, criterion, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            out = model(data)
            target = data.y.view(-1)
            loss = criterion(out, target)

            _, predicted = torch.max(out, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            running_loss += loss.item()

    test_accuracy = correct / total
    average_loss = running_loss / len(test_loader)

    return test_accuracy, average_loss

# Train the model
num_epochs = 20
for epoch in range(1, num_epochs + 1):
    train_acc, train_loss = train(model, optimizer, criterion, train_loader, device)
    test_acc, test_loss = test(model, criterion, test_loader, device)
    print(f"Epoch {epoch}, Train Accuracy: {train_acc:.4f}, Train Loss: {train_loss:.4f}, Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")

# Overall train and test accuracy
train_acc, train_loss = train(model, optimizer, criterion, train_loader, device)
test_acc, test_loss = test(model, criterion, test_loader, device)
print(f"Overall Training Accuracy: {train_acc:.4f}, Training Loss: {train_loss:.4f}")
print(f"Overall Testing Accuracy: {test_acc:.4f}, Testing Loss: {test_loss:.4f}")

# Predict function for a single image
def predict_sample_image(image_path, model, label_encoder, scaler, img_size=128):
    img_array = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img_array is None:
        print(f"Error: Unable to load image at {image_path}")
        return

    features = extract_features(img_array, img_size)
    features_scaled = scaler.transform(features.reshape(1, -1))

    data = create_graph_data(features_scaled, [0], k=8).to(device)
    model.eval()
    with torch.no_grad():
        out = model(data)
        predicted_label = label_encoder.inverse_transform([out.argmax(dim=1).item()])

    print(f"The predicted class for the image is: {predicted_label[0]}")

# Predict the quality of a sample image
sample_image_path = r"//img path"
predict_sample_image(sample_image_path, model, label_encoder, scaler)


Epoch 1, Train Accuracy: 0.0673, Train Loss: 2.6198, Test Accuracy: 0.1850, Test Loss: 121.1532
Epoch 2, Train Accuracy: 0.2000, Train Loss: 111.1471, Test Accuracy: 0.2317, Test Loss: 94.6180
Epoch 3, Train Accuracy: 0.2385, Train Loss: 88.0127, Test Accuracy: 0.3121, Test Loss: 73.5641
Epoch 4, Train Accuracy: 0.3243, Train Loss: 68.3303, Test Accuracy: 0.3325, Test Loss: 58.4330
Epoch 5, Train Accuracy: 0.3433, Train Loss: 54.5602, Test Accuracy: 0.3583, Test Loss: 44.5369
Epoch 6, Train Accuracy: 0.3796, Train Loss: 40.3707, Test Accuracy: 0.3358, Test Loss: 45.7047
Epoch 7, Train Accuracy: 0.3619, Train Loss: 38.9313, Test Accuracy: 0.3688, Test Loss: 38.7108
Epoch 8, Train Accuracy: 0.3969, Train Loss: 32.4855, Test Accuracy: 0.3571, Test Loss: 33.7522
Epoch 9, Train Accuracy: 0.3957, Train Loss: 27.2446, Test Accuracy: 0.4562, Test Loss: 25.7392
Epoch 10, Train Accuracy: 0.5084, Train Loss: 20.7088, Test Accuracy: 0.5133, Test Loss: 20.7544
Epoch 11, Train Accuracy: 0.5669, Trai